In [1]:
import pandas as pd
import gspread
from oauth2client.service_account import ServiceAccountCredentials
import pandas as pd
import os.path,sys
# import streamlit as st
# from st_aggrid import AgGrid, GridUpdateMode
# from st_aggrid.grid_options_builder import GridOptionsBuilder
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', None)

/home/joe/anaconda3/envs/bic/lib/python3.6/site-packages/google/auth/crypt/_cryptography_rsa.py:22: CryptographyDeprecationWarning: Python 3.6 is no longer supported by the Python core team. Therefore, support for it is deprecated in cryptography. The next release of cryptography will remove support for Python 3.6.
  import cryptography.exceptions


## Read Xentity's CIM Inventory

In [2]:
df1 = pd.read_csv("Asset_Inventory.csv")
print(df1.shape)

(2556, 111)


In [3]:
dfCIM = df1.loc[df1["Owner"] == "Colorado Information Marketplace"]
idsCim = dfCIM["UID"].values.tolist()

In [4]:
print(dfCIM.shape,len(idsCim))

(534, 111) 534


In [5]:
print(dfCIM["Type"].value_counts())
print(dfCIM["Audience"].value_counts())

dataset     347
map         162
href         15
filter        7
datalens      3
Name: Type, dtype: int64
public     466
private     68
Name: Audience, dtype: int64


### Extract on Map datasets that are Public

In [6]:
#  Extract only CIM datasets that are maps and public
dfCIMMap = dfCIM.loc[(dfCIM["Type"] == "map") & (dfCIM["Audience"] == "public")]
idsCimMaptmp = dfCIMMap["UID"].values.tolist()
idsCimMap = []

for id in idsCimMaptmp:
    spl = id.split(":")
    idsCimMap.append(spl[0])

In [7]:
print(len(idsCimMap))
print(dfCIMMap.shape)

157
(157, 111)


In [8]:
# for id in sorted(idsCimMap):
#     print(id)

### Split for Census and Non Census data

In [8]:
dfCIMMapCen = dfCIMMap.loc[dfCIMMap["Name"].str.lower().str.contains("census")]
dfCIMMapNoCen = dfCIMMap.loc[~dfCIMMap["Name"].str.lower().str.contains("census")]

In [10]:
print(dfCIMMap.shape)
print(dfCIMMapCen.shape)
print(dfCIMMapNoCen.shape)

(157, 111)
(92, 111)
(65, 111)


## Tyler Inventory

In [11]:
dfYvn = pd.read_csv("Mondara_Maps.csv")

In [12]:
print(dfYvn.shape)
print(dfCIMMap.shape)

(160, 7)
(157, 111)


In [13]:
idsYvn = dfYvn["uid"].values.tolist()

## Compare Xentity Inventory to Tylers (Yvonne's)

dfCIMMap  -> contains all records that are CIM only, are Maps and Public
dfYvn     -> contains Tyler (Yvonnes) inventory records

idsYvn    -> Tyler ids
idsCimMap -> ids for dfCIMMap

dfCIMMapNoCen ->  all dfCIMMap records that are NOT Census related
dfCIMMapCen   ->  all dfCIMMap records related to the Census 

In [14]:
#  datasets in our inventory not in Tylers
for index,row in dfCIMMap.iterrows():
    id=row["UID"]
    spl = id.split(":")
    idn = spl[0]
    if idn not in idsYvn:
        print(idn,row["Name"])

25ry-d5r8 Highway Traffic Counts in Colorado 2017 Map
uman-zjh4 Census in Colorado 2017 Map
azt3-ee77 Census in Colorado 2018 Map
dqd2-3hbj Points of Interest in Denver - Map
mmq3-xrh7 Tree Canopy Assesment Denver - Map
v328-h7vn Map of Recently Approved Liquor Licenses in Colorado
92n9-xx4r Map of Liquor Licenses in Colorado
5jme-s3qi Census Places in Colorado 2018 Map
8q7d-v9sc Census Congressional Districts in Colorado 2018 Map
8bxs-9ctd Census Congressional Districts in Colorado 2019 Map
i6vs-j5v9 Census Tracts in Colorado 2019 Map
pkue-zrfg Business Improvement Districts in Denver Map
r8uq-quqp Census Tracts in Colorado 2018 Map
fwyp-a5y7 EV Charging and Alternative Fuels Stations in Colorado Map
ibr3-r5n3 Census Places in Colorado 2017 Map
fcvc-7tkz Census Places in Colorado 2019 Map
w9zf-5neg Highway Traffic Counts in Colorado 2016 Map
es4r-wt2k Census Block Groups in Colorado 2018 Map
g7tq-rkfe Map of Sales Rooms in Colorado
ewqz-5hmi Scenic Byways in Colorado Map
rm7t-rch9 Cen

In [15]:
#  Datasets in Tyler list NOT IN our inventory
nmiss=0
owners = {}
for index,row in dfYvn.iterrows():
    id=row["uid"]
    if id not in idsCimMap:
         nmiss+=1
         tmp = df1.loc[df1["UID"] == id,"Owner"].values.tolist()
         for own in tmp:
             if own not in owners:
                owners[own]=0
             owners[own]+= 1
         print(id,row["name"])
         # print(tmp)
         # print("-------------------------")

38bt-frhw ADA Curb Ramp Inventory
eqi3-a27g ADA Curb Ramp Inventory
au67-c4ig All Haz Reg Test Split
p3i7-jjzd CDOT Curb Ramp Inventory: ADA Compliance Factors
szhr-pc3j CDPHE MAP PACKAGE HFACILITIES 2013
w4kj-bsss CO County Boundry
dbeg-r7bz Census Zipcodes 2010
squa-cqq6 Cities
cz3y-bbuv Colo Counties
nvz8-4p3b Colorado Counties (1)
434q-2p8p Colorado Counties Colors
dxhx-4q4y Colorado Counties
p753-497k Colorado Counties
ygbf-cga5 Colorado Counties
jvug-2qau Comminity Anchor Institutions in Colorado 2016
ysa7-n95j Community Anchor Institutions 2017
b6x9-m3zy DOLA Housing Project
6g4b-w8bd Denver Statistical Neighborhoods
nsba-2nqt Garfield County Boundary
seg6-525n Garfield County Incorporated Cities & Towns
ai9y-w2g5 Garfield County Major Rivers & Bodies of Water
7548-bu6k Garfield County Parcels
7bpv-6pjt Garfield County Points Of Interest
wair-rh7k Garfield County Public Buildings
b8i8-bzje Garfield County Roads
qkcy-pt7c Garfield County Structures
rrtm-iex5 Hydrology Basins Phas

In [16]:
owners

{'Steve Ruge - CDOT': 3,
 'Dan Martinez': 9,
 'camillid': 4,
 'April Broome': 1,
 'Jack Reed': 2,
 'Tudor Stanescu': 1,
 'OIT - GIS': 4,
 'mrmspyker': 1,
 'Garfield County GIS': 8,
 'Adrian': 1,
 'Colorado Information Marketplace': 2,
 'Jessica Potter': 2,
 'Brandon.Serna': 1}

## Result # 1

 Of the 161 datasets, 37 or now owned by CIM.  None of the 37 are related to the Census, so they would be from the 65 Non Census datasets I would send.  Therefore, I will send our list instead of Yvonnes and note it to them. 

In [17]:
dfCIMMapNoCen.columns

Index(['UID', 'Name', 'Description', 'Owner', 'Owner UID', 'Publication Stage',
       'Audience', 'Approval Status', 'Provenance', 'Type',
       ...
       'Data Quality: Stewardship Plan', 'Data Updates: Update Schedule',
       'Data Updates: Total Columns at Recent Update',
       'Dataset Summary: Progress',
       'Additional Dataset Documentation: Data Dictionary',
       'Department Metrics: Publishing Department',
       'Data Description: Oldest Record in Dataset',
       'Dataset Summary: Sub Agency',
       'Data Description: Date of Initial Dataset Creation',
       'Data Description: Collection Mode'],
      dtype='object', length=111)

In [18]:
dfCIMMapNoCen[['UID', 'Name', 'Owner','Audience','Type']].to_csv("Map_Datasets_NoCensus.csv",index=False)

## Census Datasets

In [19]:
dfCIMMapCen.shape

(92, 111)

In [21]:
def getFields(w4x4=""):
    '''Reads the Inventory google sheet and getsthe fields by 4x4 dataset id and by the title'''
    bic_etl_home = os.getenv('bic_etl_home')
    print("------------")
    print("Get Fields starting",bic_etl_home)
    print("------------")

## Add the bic_etl/general/script directory to path 
    sys.path.insert(0, os.path.join(bic_etl_home, 'general', 'scripts'))

    SERVICE_ACCOUNT_FILE = os.path.join("/home","joe", 'work',"client_secret.json")
    print(SERVICE_ACCOUNT_FILE)
    scope = ['https://www.googleapis.com/auth/spreadsheets.readonly',
             "https://www.googleapis.com/auth/drive"]

    creds = ServiceAccountCredentials.from_json_keyfile_name(SERVICE_ACCOUNT_FILE,
     scope)
    client = gspread.authorize(creds)

    fields_sheet = client.open('BIC Data Inventory and Metadata').worksheet(
        'MetadataRepository')
    
    print("field sheet",fields_sheet)

    dfFields = pd.DataFrame(fields_sheet.get_all_records(head=3))
    
    fields = {}
    for index,row in dfFields.iterrows():
        s4x4 = row["Socrata Link"].strip()
        a4x4 = row["API 4x4"].strip()
        fields[s4x4] = a4x4
  
    return fields

fields = getFields()

------------
Get Fields starting /home/joe/bic_etl
------------
/home/joe/work/client_secret.json
field sheet <Worksheet 'MetadataRepository' id:323813469>


In [37]:
errs = {}
no = []
yes = []
data = {}
nrows = dfCIMMapCen.shape[0]
count=0
for index,row in dfCIMMapCen.iterrows():
   count+=1
   id = row["UID"]
   name = row["Name"]
   if id in fields:
        api = fields[id]
        print("API ",api)
   else:
        api = id
        print("NO API",api)
   string = f"https://data.colorado.gov/api/views/{api}/rows.csv?accessType=DOWNLOAD"
   print(name)
   print(f"{count} of {nrows}: {string}")
   try:
     df = pd.read_csv(string)
     yes.append(id)
     print("hit",len(yes))
     data[id] = {}
     data[id]['name'] = name
     data[id]['shape'] = df.shape
    
    
     data[id]['data'] = df.head()
     print("-------------------\n")
        
   except Exception as err:
      no.append(id)
      errs[id] = err
      print(err)
      print("------------------------------------\n")
   

API  kt3p-393c
Census Places in Colorado 2012
1 of 92: https://data.colorado.gov/api/views/kt3p-393c/rows.csv?accessType=DOWNLOAD
hit 1
-------------------

API  rt3s-2378
Census Tracts in Colorado 2010
2 of 92: https://data.colorado.gov/api/views/rt3s-2378/rows.csv?accessType=DOWNLOAD
hit 2
-------------------

API  eghh-ua8y
Census Counties in Colorado 2016
3 of 92: https://data.colorado.gov/api/views/eghh-ua8y/rows.csv?accessType=DOWNLOAD
hit 3
-------------------

NO API uman-zjh4
Census in Colorado 2017 Map
4 of 92: https://data.colorado.gov/api/views/uman-zjh4/rows.csv?accessType=DOWNLOAD
No columns to parse from file
------------------------------------

NO API azt3-ee77
Census in Colorado 2018 Map
5 of 92: https://data.colorado.gov/api/views/azt3-ee77/rows.csv?accessType=DOWNLOAD
No columns to parse from file
------------------------------------

API  n5xr-summ
Census Places in Colorado 2014
6 of 92: https://data.colorado.gov/api/views/n5xr-summ/rows.csv?accessType=DOWNLOAD
hit

In [27]:
no

['uman-zjh4',
 'azt3-ee77',
 '5jme-s3qi',
 '8q7d-v9sc',
 '8bxs-9ctd',
 'i6vs-j5v9',
 'r8uq-quqp',
 'ibr3-r5n3',
 'fcvc-7tkz',
 'es4r-wt2k',
 'vne4-cwmy',
 'rwak-e74e',
 'rm7t-rch9',
 'au4m-9k5p',
 '3nde-6qne',
 'kje4-ci7c',
 'kzrg-3rgm',
 'ssuw-z5kg',
 'dms8-ujea',
 'c3rd-t2me',
 'gbbd-diue']

In [28]:
fields

{'rifs-n6ib': 'rifs-n6ib',
 'c8jj-hcxj': 'nigi-mnyx',
 '82s5-cpkk': '82s5-cpkk',
 'n55r-9hud': 'n55r-9hud',
 'ms6b-y4xc': 'ms6b-y4xc',
 'e6e8-qmi7': 'px7t-j3xt',
 'hxf8-ab6k': 'yt5k-hawq',
 'p8m4-v33g': 'vgwh-ygn3',
 'g53r-j5td': '6pd5-jdur',
 'jnj7-fw37': 'jnj7-fw37',
 'gyeb-jc69': '9gxp-sqz6',
 'u2t6-bfhr': 'iw7e-tqkz',
 'cjkq-q9ih': 'iaz7-gn2p',
 '4e3w-qire': 'da3r-y7xz',
 'busm-qa5b': 'vu7j-izta',
 '2cpa-vbur': 'udxx-a7kq',
 'bu8h-8sux': 'suda-myfn',
 'pt2g-89wc': '7rd3-4hgz',
 'bynd-i2hj': 'cy35-ndjp',
 'k3gg-hhc8': 'k3gg-hhc8',
 'x8tb-f3vh': 'x8tb-f3vh',
 '2yhn-3dbj': '2yhn-3dbj',
 '6kn4-89kh': '6kn4-89kh',
 '869n-zj3f': '869n-zj3f',
 'sqs8-2un5': 'sqs8-2un5',
 '3sm5-jtur': '3sm5-jtur',
 'ehk3-i5tr': 'ehk3-i5tr',
 'v9m8-x8dh': 'v9m8-x8dh',
 'j7a3-jgd3': 'j7a3-jgd3',
 'kapc-ib6e': 'ii5c-5549',
 'htyp-tqzh': 'htyp-tqzh',
 'ier5-5ms2': 'ier5-5ms2',
 'pwjb-9dd5': 'pwjb-9dd5',
 '9pwz-gi5v': 'ic4i-9zku',
 'd4s4-xqg6': '394i-tr5f',
 'msap-49q7': 'msap-49q7',
 'wz8h-dap6': 'wz8h-dap6',
 

In [33]:
for col in sorted(dfCIMMapCen.columns):
    print(col)

API Endpoint
Additional Dataset Documentation: Additional Metadata
Additional Dataset Documentation: Data Dictionary
Additional Dataset Documentation: FGDC Compliance (Geospatial Only)
Additional Dataset Documentation: Technical Documentation
Approval Status
Attribution
Attribution Link
Audience
Category
Column Count
Contact Email
Contributing Agency Information: Agency Data Series Page
Contributing Agency Information: Agency Program Page
Contributing Agency Information: Business Contact and Phone
Contributing Agency Information: Citation
Contributing Agency Information: Data Source
Contributing Agency Information: Technical Contact and Phone
Creation Date (UTC)
Data Description: Collection Method
Data Description: Collection Mode
Data Description: Data Collection Instrument
Data Description: Date of Initial Dataset Creation
Data Description: Field Names, comma delimited
Data Description: Long Description
Data Description: Newest Record in Dataset
Data Description: Oldest Record in Dat

In [35]:

print(dfCIMMapCen.loc[dfCIMMapCen["UID"].isin(no),["Name","Audience"]])

                                                   Name Audience
370                         Census in Colorado 2017 Map   public
371                         Census in Colorado 2018 Map   public
789                  Census Places in Colorado 2018 Map   public
983   Census Congressional Districts in Colorado 201...   public
986   Census Congressional Districts in Colorado 201...   public
1027                 Census Tracts in Colorado 2019 Map   public
1075                 Census Tracts in Colorado 2018 Map   public
1248                 Census Places in Colorado 2017 Map   public
1250                 Census Places in Colorado 2019 Map   public
1289           Census Block Groups in Colorado 2018 Map   public
1413                  Census Zip Codes in Colorado 2014   public
1425                  Census Zip Codes in Colorado 2016   public
1462               Census Counties in Colorado 2019 Map   public
1479              Census Zip Codes in Colorado 2018 Map   public
1658               Census

In [ ]:
data

In [90]:
for id,dct in data.items():
    print(id,dct["shape"],dct["name"])
    display(dct["data"].head(3))
    
    

n53p-giqe (455, 157) Census Places in Colorado 2012


,the_geom,pop,geonum,geoname,hispanic,white_nh,black_nh,ntvam_nh,asian_nh,hawpi_nh,other_nh,twoplus_nh,male,female,ageless5,age5_9,age10_14,age15_19,age20_24,age25_29,age30_34,age35_39,age40_44,age45_49,age50_54,age55_59,age60_64,age65_69,age70_74,age75_79,age80_84,age85pl,ageless18,age18_24,med_age,households,familyhh,nonfamhh,hhldralone,hhldr_naln,housing_un,occ_hu,vac_hu,owned,rented,pop25plus,nohsdipl,hsgrad_sc,bachl_hghr,med_hh_inc,med_fam_in,per_cap_in,med_yr_blt,med_c_rent,med_g_rent,med_hm_val,citz_birth,citz_nat,not_citz,born_in_co,brn_oth_st,ntv_b_o_us,foreign_b,pop_1p,same_house,same_cnty,same_state,diff_state,frm_abroad,wrkrs_16pl,car_all,car_alone,car_carpoo,public_trn,pt_bus,pt_other,bike,walk,tr_other,wrk_home,w_16pl_nh,t_less_10,t_10_19,t_20_29,t_30_39,t_40_59,t_60_pl,pop_3pl,enrolled,preschool,kndrgrtn,gr_1_4,gr_5_8,gr_9_12,undergrad,grad_prof,n_enrolled,blt_2010_p,b2000_2009,b1990_1999,b1980_1989,b1970_1979,b1960_1969,b1950_1959,b1940_1949,b1939_e,own_occ_hu,v_l_50k,v50k_100k,v100k_150k,v150k_200k,v200k_250k,v250k_300k,v300k_400k,v400k_500k,v500k_750k,v750k_1m,v_1m_plus,rnt_occ_hu,r400t599,rntl400,r600t799,r800t999,r1000t1249,r1250t1499,r1500t1999,r2000pl,rnocshr,ps_uni,ps_below,ps_bel150,tot_l18,puni_65pl,pov_l18,pov_65pl,avghhsize,hhi_l20k,hhi20_30,hhi30_40,hhi40_50,hhi50_60,hhi60_75,hhi75_100,hhi100_125,hhi125_150,hhi150_200,hhi200_pl,civ_ni_pop,disabled,pop16_pls,laborforce,civ_lf,emp,unemp,armedfrcs,not_lf,civ_ni_p
0,"MULTIPOLYGON (((-103.848012 38.194975, -103.86...",349,10818750,Crowley,241,104,4,0,0,0,0,0,183,166,27,8,0,32,30,50,22,0,10,54,27,11,26,18,13,2,10,9,51,46,42.3,94,73,21,16,5,106,94,12,81,13,252,41,147,64,45326.0,45924.0,15623.0,1939.0,241.0,389.0,62500.0,347,0,2,254,93,0,2,345,312,3,18,12,0,148,145,113,32,0,0,0,0,2,0,1,147,13,73,30,0,25,6,331,72,5,0,8,0,32,27,0,259,0,3,10,8,10,0,1,14,60,81,30,43,3,3,0,0,0,0,0,2,0,13,2,8,0,3,0,0,0,0,0,346,29,61,48,52,9,0,3.71,12,17,14,30,5,14,2,0,0,0,0,349,64,306,196,196,148,48,0,110,349
1,MULTIPOLYGON (((-104.930313 39.856173999999996...,118747,10877290,Thornton,35788,73220,1747,375,5078,105,364,2070,59356,59391,9524,10910,9588,7589,7734,9579,10457,8963,9733,8772,7386,6262,4149,2966,1923,1221,964,1027,34907,10438,31.9,40253,29613,10640,8256,2384,42954,40253,2701,28624,11629,73402,9172,45285,18945,66176.0,74246.0,26566.0,1992.0,884.0,1046.0,207600.0,106292,5168,7287,60826,44063,1403,12455,116959,98496,9641,6093,2213,516,59688,54341,47941,6400,1944,1938,6,222,601,461,2119,57569,3911,11757,13333,13631,11036,3901,113366,33810,2360,2184,8037,7697,6849,5468,1215,79556,44,13382,10495,6523,7992,1579,2639,88,212,28624,2235,779,3297,6951,6950,4415,3206,552,76,51,112,11629,321,294,1686,2840,3199,1596,1280,241,172,117580,10796,19702,34179,7881,4580,425,2.94,3661,3032,3540,4018,3738,5064,6528,4972,2817,1902,981,118129,9129,87086,65525,65349,60590,4759,176,21561,118129
2,"MULTIPOLYGON (((-105.224318 40.057601, -105.21...",99177,10807850,Boulder,8742,82648,786,112,4424,52,181,2232,49955,49222,4194,4057,3732,11447,19882,8680,6401,6197,5833,5449,5063,5298,4022,2710,1807,1557,1253,1595,14190,29122,28.4,41076,17653,23423,13642,9781,43322,41076,2246,20000,21076,55865,2383,13670,39812,56206.0,102379.0,37734.0,1974.0,1061.0,1132.0,489500.0,88743,3851,6583,26605,60707,1431,10434,98244,61498,19271,6827,8835,1813,52461,30663,27546,3117,4679,4651,28,5507,4813,724,6075,46386,9404,20219,7677,3569,3592,1925,96756,40780,1528,794,3206,2952,2573,24986,4741,55976,209,3646,4519,7101,10434,8303,4398,938,3774,20000,898,303,584,991,965,1030,2648,2889,5721,2284,1687,21076,522,648,2348,4677,3921,2636,3434,2349,541,90911,19625,28195,14003,8356,1541,508,2.20,8727,3619,3704,2878,2470,3644,4111,2944,2221,2900,3858,97859,6321,86663,57908,57810,53483,4327,98,28755,97859


gu6r-zvv2 (1249, 77) Census Tracts in Colorado 2010


,the_geom,NAME10,NAMELSAD10,AREALAND,AREAWATR,INTPTLAT,INTPTLON,STATE,COUNTY,TRACT,GEOIDTRACT,POP2010,HISPANIC,PCT_HISP,WHITE_NH,PCT_WHITE,BLACK_NH,PCT_BLACK,AMERIND_NH,PCT_AMIND,ASIAN_NH,PCT_ASIAN,HAWPAC_NH,PCT_HAWPI,OTHER_NH,PCT_OTHER,MULT_NH,PCT_MULT,MALE,PCT_MALE,FEMALE,PCT_FEM,AGE65PLUS,PCT_65PLUS,AGEUNDER18,PCT_U18,AGEUNDER10,PCT_U10,MED_AGE,MED_MALE,MED_FEMALE,HOUSEHOLDS,FAMILY_HH,HUSB_WIFE,OTH_FAMILY,NON_FAM,HH_ALONE,AVG_HH,HH_W_NONR,HH_N_NONR,AVG_FAM,GQ_POP,GQ_INST,GQ_CORREC,GQ_JUVENL,GQ_NURS,GQ_OT_INST,GQ_NONINST,GQ_COLLEGE,GQ_MILTRY,GQ_O_NONI,HOUSING_UN,OCCUPIED,PCT_OCC,VACANT,PCT_VAC,OWNER,PCT_OWN,RENTER,PCT_RENT,V_FORRENT,V_R_NOTOCC,V_FORSALE,V_S_NOTOCC,V_SEASONAL,V_MIGRANT,V_OTHER
0,MULTIPOLYGON (((-107.12846200000011 38.8178610...,9639.0,Census Tract 9639,4262312150,41657300,38.628582,-107.273386,8,51,963900,8051963900,905,36,3.98,847,93.59,1,0.11,8,0.88,2,0.22,0,0.00,0,0.00,11,1.22,480,53.04,425,46.96,146,16.13,154,17.02,82,9.06,49.6,50.3,48.9,390,256,231,25,134,104,2.19,38,352,2.64,51,0,0,0,0,0,51,13,0,38,1348,390,28.93,958,71.07,308,78.97,82,21.03,9,0,22,0,880,1,46
1,MULTIPOLYGON (((-106.9526920000001 38.55070300...,9637.0,Census Tract 9637,14502053,0,38.544337,-106.927026,8,51,963700,8051963700,6124,844,13.78,5060,82.63,35,0.57,26,0.42,34,0.56,2,0.03,13,0.21,110,1.80,3379,55.18,2745,44.82,487,7.95,973,15.89,577,9.42,25.6,24.7,27.7,2446,1063,774,289,1383,853,2.19,625,1821,2.87,760,48,3,0,45,0,712,712,0,0,2784,2446,87.86,338,12.14,1032,42.19,1414,57.81,163,7,26,3,67,0,72
2,MULTIPOLYGON (((-106.5182840000001 38.95151700...,9636.0,Census Tract 9636,3267477050,10725478,38.673896,-106.709626,8,51,963600,8051963600,3794,231,6.09,3491,92.01,7,0.18,12,0.32,20,0.53,0,0.00,0,0.00,33,0.87,1993,52.53,1801,47.47,469,12.36,769,20.27,411,10.83,43.3,42.9,43.8,1628,1083,949,134,545,429,2.31,162,1466,2.77,35,0,0,0,0,0,35,0,0,35,3130,1628,52.01,1502,47.99,1274,78.26,354,21.74,24,3,55,9,1333,1,77


sn6p-34bq (64, 157) Census Counties in Colorado 2016


...

,pop,the_geom,geoname,geonum,hispanic,white_nh,black_nh,ntvam_nh,asian_nh,hawpi_nh,other_nh,twoplus_nh,male,female,ageless5,age5_9,age10_14,age15_19,age20_24,age25_29,age30_34,age35_39,age40_44,age45_49,age50_54,age55_59,age60_64,age65_69,age70_74,age75_79,age80_84,age85pl,ageless18,age18_24,med_age,households,familyhh,nonfamhh,hhldralone,hhldr_naln,housing_un,occ_hu,vac_hu,owned,rented,pop25plus,nohsdipl,hsgrad_sc,bachl_hghr,med_hh_inc,med_fam_in,per_cap_in,med_yr_blt,med_c_rent,med_g_rent,med_hm_val,citz_birth,citz_nat,not_citz,born_in_co,brn_oth_st,ntv_b_o_us,foreign_b,pop_1p,same_house,same_cnty,same_state,diff_state,frm_abroad,wrkrs_16pl,car_all,car_alone,car_carpoo,public_trn,pt_bus,pt_other,bike,walk,tr_other,wrk_home,w_16pl_nh,t_less_10,t_10_19,t_20_29,t_30_39,t_40_59,t_60_pl,pop_3pl,enrolled,preschool,kndrgrtn,gr_1_4,gr_5_8,gr_9_12,undergrad,grad_prof,n_enrolled,blt_2010_p,b2000_2009,b1990_1999,b1980_1989,b1970_1979,b1960_1969,b1950_1959,b1940_1949,b1939_e,own_occ_hu,v_l_50k,v50k_100k,v100k_150k,v150k_200k,v200k_250k,v250k_300k,v300k_400k,v400k_500k,v500k_750k,v750k_1m,v_1m_plus,rnt_occ_hu,rntl400,r400t599,r600t799,r800t999,r1000t1249,r1250t1499,r1500t1999,r2000pl,rnocshr,ps_uni,ps_below,ps_bel150,tot_l18,puni_65pl,pov_l18,pov_65pl,avghhsize,hhi_l20k,hhi20_30,hhi30_40,hhi40_50,hhi50_60,hhi60_75,hhi75_100,hhi100_125,hhi125_150,hhi150_200,hhi200_pl,civ_ni_pop,disabled,pop16_pls,laborforce,civ_lf,emp,unemp,armedfrcs,not_lf,civ_ni_p
0,5197580,"MULTIPOLYGON (((-109.060253 38.599328, -109.05...",Colorado,108,1088602,3605199,198952,28452,143581,5921,7963,118910,2608270,2589310,337435,352695,344661,341735,367651,385593,378791,355630,355020,351550,372616,344010,295848,214879,144269,105138,74722,75337,1234697,509480,36.2,1998314,1282425,715889,560361,155528,2238624,1998314,240310,1295048,703266,3453403,332246,1827473,1293684,59448,73817,31674,1981,848,969,239400,4690377,192391,314812,2212968,2402164,75245,507203,5133849,4139324,497646,257863,203700,35316,2544021,2157410,1907274,250136,82367,70380,11987,33553,76376,27947,166368,2377653,328644,720723,517659,396871,259022,154734,4998019,1388718,88938,70606,279389,275611,271312,325375,77487,3609301,23814,429203,378604,334185,428767,209978,182318,64634,187121,1295048,70143,67138,136423,215484,200845,155554,197540,99600,96774,28641,26906,703266,36922,55774,127783,139089,128861,82817,76375,29764,25881,5079529,663861,1095466,1218579,598714,208902,45544,2.54,295537,188690,185732,176480,160263,204968,263329,180627,112904,119480,110304,5105164,526054,4094922,2811701,2779944,2560703,219241,31757,1283221,5105164


jd2b-7gyb (526, 156) Census Zip Codes in Colorado 2015


,geoid10,hispanic,the_geom,pop,white_nh,black_nh,ntvam_nh,asian_nh,hawpi_nh,other_nh,twoplus_nh,male,female,ageless5,age5_9,age10_14,age15_19,age20_24,age25_29,age30_34,age35_39,age40_44,age45_49,age50_54,age55_59,age60_64,age65_69,age70_74,age75_79,age80_84,age85pl,ageless18,age18_24,med_age,households,familyhh,nonfamhh,hhldralone,hhldr_naln,housing_un,occ_hu,vac_hu,owned,rented,pop25plus,nohsdipl,hsgrad_sc,bachl_hghr,med_hh_inc,med_fam_in,per_cap_in,med_yr_blt,med_c_rent,med_g_rent,med_hm_val,citz_birth,citz_nat,not_citz,born_in_co,brn_oth_st,ntv_b_o_us,foreign_b,pop_1p,same_house,same_cnty,same_state,diff_state,frm_abroad,wrkrs_16pl,car_all,car_alone,car_carpoo,public_trn,pt_bus,pt_other,bike,walk,tr_other,wrk_home,w_16pl_nh,t_less_10,t_10_19,t_20_29,t_30_39,t_40_59,t_60_pl,pop_3pl,enrolled,preschool,kndrgrtn,gr_1_4,gr_5_8,gr_9_12,undergrad,grad_prof,n_enrolled,blt_2010_p,b2000_2009,b1990_1999,b1980_1989,b1970_1979,b1960_1969,b1950_1959,b1940_1949,b1939_e,own_occ_hu,v_l_50k,v50k_100k,v100k_150k,v150k_200k,v200k_250k,v250k_300k,v300k_400k,v400k_500k,v500k_750k,v750k_1m,v_1m_plus,rnt_occ_hu,rntl400,r400T599,r600T799,r800t999,r1000t1249,r1250t1499,r1500t1999,r2000pl,rnocshr,ps_uni,ps_below,ps_bel150,tot_l18,puni_65pl,pov_l18,pov_65pl,avghhsize,hhi_l20k,hhi20_30,hhi30_40,hhi40_50,hhi50_60,hhi60_75,hhi75_100,hhi100_125,hhi125_150,hhi150_200,hhi200_pl,civ_ni_pop,disabled,pop16_pls,laborforce,civ_lf,emp,unemp,armedfrcs,not_lf,civ_ni_p
0,80476,5,MULTIPOLYGON (((-105.92508499987281 39.6794199...,253,247,0,0,0,0,0,1,139,114,10,5,9,11,4,18,14,21,13,34,15,22,19,24,13,21,0,0,31,8,47.0,125,66,59,38,21,206,125,81,87,38,214,4,131,79,46250.0,79167.0,33168.0,1939.0,681.0,863.0,183700.0,250,0,3,87,160,3,3,253,199,34,13,7,0,131,111,107,4,3,3,0,5,9,0,3,128,42,25,20,6,23,12,246,33,3,1,4,7,13,5,0,213,0,7,10,9,10,21,22,0,127,87,0,2,23,27,10,7,5,13,0,0,0,38,0,7,4,17,6,3,0,0,1,249,27,62,27,58,0,3,2.02,13,22,20,12,1,16,8,10,12,11,0,253,16,224,156,156,149,7,0,68,253
1,80477,117,MULTIPOLYGON (((-106.82821600036812 40.4871999...,273,156,0,0,0,0,0,0,158,115,0,0,51,75,0,15,0,29,0,29,12,20,5,0,21,16,0,0,78,48,28.7,93,77,16,16,0,93,93,0,64,29,147,0,61,86,NaN,NaN,46049.0,1949.0,NaN,NaN,847200.0,273,0,0,123,150,0,0,273,185,88,0,0,0,150,32,32,0,77,77,0,33,0,8,0,150,38,27,29,48,8,0,273,78,0,0,0,51,27,0,0,195,0,8,0,37,0,0,0,26,22,64,0,0,0,0,0,0,8,0,17,18,21,29,0,0,0,29,0,0,0,0,0,273,9,9,78,37,0,0,2.94,0,0,8,29,0,0,5,0,10,12,29,273,48,222,150,150,150,0,0,72,273
2,80478,0,MULTIPOLYGON (((-105.91747200043244 40.0054129...,1541,1416,15,0,102,0,0,8,894,647,24,0,41,131,318,139,34,161,41,195,131,100,51,175,0,0,0,0,150,364,36.5,442,282,160,136,24,1120,442,678,382,60,1027,0,532,495,87581.0,88072.0,28960.0,1998.0,NaN,NaN,NaN,1390,8,143,249,1141,0,151,1517,1019,67,112,310,9,1093,964,430,534,6,6,0,0,73,0,50,1043,610,296,36,0,7,94,1517,163,0,0,0,41,91,31,0,1354,35,499,128,39,309,66,0,0,44,382,0,44,0,38,130,0,0,0,128,0,42,60,0,0,0,0,36,0,0,24,0,1541,459,467,150,175,55,0,3.24,82,0,0,51,0,48,160,77,0,24,0,1541,0,1446,1157,1157,1113,44,0,289,1541


wcvg-8vki (526, 73) Census Zip Codes in Colorado 2010


,the_geom,NAME10,ZIPCODE,AREALAND,AREAWATR,INTPTLAT,INTPTLON,POP2010,HISPANIC,PCT_HISP,WHITE_NH,PCT_WHITE,BLACK_NH,PCT_BLACK,AMERIND_NH,PCT_AMIND,ASIAN_NH,PCT_ASIAN,HAWPAC_NH,PCT_HAWPI,OTHER_NH,PCT_OTHER,MULT_NH,PCT_MULT,MALE,PCT_MALE,FEMALE,PCT_FEM,AGE65PLUS,PCT_65PLUS,AGEUNDER18,PCT_U18,AGEUNDER10,PCT_U10,MED_AGE,MED_MALE,MED_FEMALE,HOUSEHOLDS,FAMILY_HH,HUSB_WIFE,OTH_FAMILY,NON_FAM,HH_ALONE,AVG_HH,HH_W_NONR,HH_N_NONR,AVG_FAM,GQ_POP,GQ_INST,GQ_CORREC,GQ_JUVENL,GQ_NURS,GQ_OT_INST,GQ_NONINST,GQ_COLLEGE,GQ_MILTRY,GQ_O_NONI,HOUSING_UN,OCCUPIED,PCT_OCC,VACANT,PCT_VAC,OWNER,PCT_OWN,RENTER,PCT_RENT,V_FORRENT,V_R_NOTOCC,V_FORSALE,V_S_NOTOCC,V_SEASONAL,V_MIGRANT,V_OTHER
0,MULTIPOLYGON (((-105.0089030000001 40.00029000...,ZCTA5 80023,80023,45657286,566048,39.970697,-105.017826,15415,1085,7.04,12744,82.67,142,0.92,31,0.20,1084,7.03,14,0.09,10,0.06,305,1.98,7529,48.84,7886,51.16,1705,11.06,4446,28.84,2741,17.78,39.1,38.5,39.7,5658,4400,3969,431,1258,1037,2.72,356,5302,3.12,0,0,0,0,0,0,0,0,0,0,6024,5658,93.92,366,6.08,5085,89.87,573,10.13,79,3,140,32,71,0,41
1,MULTIPOLYGON (((-104.9270340000001 39.79105400...,ZCTA5 80022,80022,119260445,708605,39.862866,-104.874466,42699,22218,52.03,17689,41.43,1177,2.76,209,0.49,667,1.56,26,0.06,68,0.16,645,1.51,21689,50.80,21010,49.20,3010,7.05,13741,32.18,8486,19.87,30.7,30.7,30.8,13380,10097,7205,2892,3283,2470,3.16,1997,11383,3.61,420,201,0,10,191,0,219,0,0,219,14343,13380,93.29,963,6.71,8855,66.18,4525,33.82,264,17,267,34,33,0,348
2,MULTIPOLYGON (((-104.2651050000001 39.72109500...,ZCTA5 80103,80103,727992115,16612998,39.724905,-104.162592,2458,225,9.15,2136,86.90,24,0.98,22,0.90,10,0.41,2,0.08,1,0.04,38,1.55,1246,50.69,1212,49.31,269,10.94,671,27.30,334,13.59,40.2,40.9,39.7,902,664,542,122,238,196,2.73,78,824,3.20,0,0,0,0,0,0,0,0,0,0,1002,902,90.02,100,9.98,681,75.50,221,24.50,9,1,26,3,9,2,50


In [64]:
print(len(yes))
print(len(no))

71
21


In [67]:
df.head(10)

,the_geom,NAME10,ZIPCODE,AREALAND,AREAWATR,INTPTLAT,INTPTLON,POP2010,HISPANIC,PCT_HISP,WHITE_NH,PCT_WHITE,BLACK_NH,PCT_BLACK,AMERIND_NH,PCT_AMIND,ASIAN_NH,PCT_ASIAN,HAWPAC_NH,PCT_HAWPI,OTHER_NH,PCT_OTHER,MULT_NH,PCT_MULT,MALE,PCT_MALE,FEMALE,PCT_FEM,AGE65PLUS,PCT_65PLUS,AGEUNDER18,PCT_U18,AGEUNDER10,PCT_U10,MED_AGE,MED_MALE,MED_FEMALE,HOUSEHOLDS,FAMILY_HH,HUSB_WIFE,OTH_FAMILY,NON_FAM,HH_ALONE,AVG_HH,HH_W_NONR,HH_N_NONR,AVG_FAM,GQ_POP,GQ_INST,GQ_CORREC,GQ_JUVENL,GQ_NURS,GQ_OT_INST,GQ_NONINST,GQ_COLLEGE,GQ_MILTRY,GQ_O_NONI,HOUSING_UN,OCCUPIED,PCT_OCC,VACANT,PCT_VAC,OWNER,PCT_OWN,RENTER,PCT_RENT,V_FORRENT,V_R_NOTOCC,V_FORSALE,V_S_NOTOCC,V_SEASONAL,V_MIGRANT,V_OTHER
0,MULTIPOLYGON (((-105.0089030000001 40.00029000...,ZCTA5 80023,80023,45657286,566048,39.970697,-105.017826,15415,1085,7.04,12744,82.67,142,0.92,31,0.20,1084,7.03,14,0.09,10,0.06,305,1.98,7529,48.84,7886,51.16,1705,11.06,4446,28.84,2741,17.78,39.1,38.5,39.7,5658,4400,3969,431,1258,1037,2.72,356,5302,3.12,0,0,0,0,0,0,0,0,0,0,6024,5658,93.92,366,6.08,5085,89.87,573,10.13,79,3,140,32,71,0,41
1,MULTIPOLYGON (((-104.9270340000001 39.79105400...,ZCTA5 80022,80022,119260445,708605,39.862866,-104.874466,42699,22218,52.03,17689,41.43,1177,2.76,209,0.49,667,1.56,26,0.06,68,0.16,645,1.51,21689,50.80,21010,49.20,3010,7.05,13741,32.18,8486,19.87,30.7,30.7,30.8,13380,10097,7205,2892,3283,2470,3.16,1997,11383,3.61,420,201,0,10,191,0,219,0,0,219,14343,13380,93.29,963,6.71,8855,66.18,4525,33.82,264,17,267,34,33,0,348
2,MULTIPOLYGON (((-104.2651050000001 39.72109500...,ZCTA5 80103,80103,727992115,16612998,39.724905,-104.162592,2458,225,9.15,2136,86.90,24,0.98,22,0.90,10,0.41,2,0.08,1,0.04,38,1.55,1246,50.69,1212,49.31,269,10.94,671,27.30,334,13.59,40.2,40.9,39.7,902,664,542,122,238,196,2.73,78,824,3.20,0,0,0,0,0,0,0,0,0,0,1002,902,90.02,100,9.98,681,75.50,221,24.50,9,1,26,3,9,2,50
3,MULTIPOLYGON (((-103.8916620000001 39.86975200...,ZCTA5 80701,80701,1573752900,5996004,40.236980,-103.807123,16042,6190,38.59,8833,55.06,689,4.29,57,0.36,101,0.63,7,0.04,19,0.12,146,0.91,8023,50.01,8019,49.99,2042,12.73,4684,29.20,2669,16.64,33.6,32.8,34.5,5718,4055,3052,1003,1663,1390,2.77,582,5136,3.30,191,176,98,0,78,0,15,0,0,15,6180,5718,92.52,462,7.48,3693,64.59,2025,35.41,175,7,82,5,45,5,143
4,MULTIPOLYGON (((-104.8469320000001 39.75488900...,ZCTA5 80010,80010,14283218,31534,39.739249,-104.864054,39728,22817,57.43,8757,22.04,5482,13.80,241,0.61,1339,3.37,113,0.28,69,0.17,910,2.29,21035,52.95,18693,47.05,2830,7.12,12197,30.70,7620,19.18,29.6,29.8,29.5,12767,8208,4925,3283,4559,3413,3.02,2465,10302,3.73,1143,826,429,27,370,0,317,0,0,317,14334,12767,89.07,1567,10.93,4693,36.76,8074,63.24,934,23,235,13,21,0,341
5,MULTIPOLYGON (((-106.03933900000011 37.4623490...,ZCTA5 81101,81101,1060256710,1615149,37.476233,-105.819178,14675,6997,47.68,7040,47.97,137,0.93,130,0.89,125,0.85,6,0.04,29,0.20,211,1.44,7356,50.13,7319,49.87,1643,11.20,3625,24.70,2160,14.72,32.3,31.3,33.6,5668,3568,2511,1057,2100,1720,2.46,669,4999,3.08,740,276,227,0,49,0,464,407,0,57,6133,5668,92.42,465,7.58,3388,59.77,2280,40.23,141,8,53,28,95,6,134
6,MULTIPOLYGON (((-104.8314850000001 39.67491300...,ZCTA5 80012,80012,20111206,0,39.700204,-104.837528,46665,15512,33.24,17133,36.71,9922,21.26,221,0.47,2216,4.75,138,0.30,92,0.20,1431,3.07,23038,49.37,23627,50.63,4790,10.26,12345,26.45,7422,15.90,32.8,31.7,34.0,18115,11043,6951,4092,7072,5754,2.56,2517,15598,3.27,302,273,18,65,190,0,29,0,0,29,19634,18115,92.26,1519,7.74,8916,49.22,9199,50.78,737,23,247,28,30,0,454
7,MULTIPOLYGON (((-104.8099460000001 39.66584500...,ZCTA5 80013,80013,36229567,0,39.659591,-104.775014,69588,12672,18.21,40506,58.21,8762,12.59,340,0.49,4415,6.34,220,0.32,130,0.19,2543,3.65,34426,49.47,35162,50.53,4137,5.94,19626,28.20,10870,15.62,33.6,32.9,34.4,24791,18148,13539,4609,6643,5138,2.81,2983,21808,3.26,26,0,0,0,0,0,26,0,0,26,26047,24791,95.18,1256,4.82,18492,74.59,6299,25.41,492,23,404,62,34,0,241
8,MULTIPOLYGON (((-102.3282360000001 37.55727400...,ZCTA5 81084,81084,6238756

In [117]:
ids = []
names = []
shapes = []
datadict = []


for id,dct in data.items():
    print(id,dct["name"],dct["shape"])
    ids.append(id)
    names.append(dct["name"])
    shapes.append(dct["shape"])
    datadict.append(dct["data"])

n53p-giqe Census Places in Colorado 2012 (455, 157)
gu6r-zvv2 Census Tracts in Colorado 2010 (1249, 77)
sn6p-34bq Census Counties in Colorado 2016 (64, 157)
9cn8-g4er Census Places in Colorado 2014 (458, 157)
gafi-svtb Census Congressional Districts in Colorado 2014 (7, 156)
qx2d-46a5 Census Core Based Statistical Area in Colorado 2011 (14, 271)
khe9-7d2c Census Zip Codes SF1 in Colorado 2000 (483, 70)
v56b-yh3u Census Core Based Statistical Area in Colorado 2012 (14, 325)
mxmm-svgz Census County Subdivisions in Colorado 2012 (209, 331)
f3th-4ste Census County Subdivisions SF3 in Colorado 2000 (208, 104)
9gri-r239 Census Block Groups in Colorado 2013 (3532, 157)
22bw-ijyi Census ACS Counties in Colorado 2011 (64, 275)
37uc-bdj5 Census Zip Codes in Colorado 2012 (526, 156)
xymp-u28i Census Counties in Colorado 2013 (64, 157)
xxfz-y2eq Census in Colorado 2015 (1, 157)
wtpp-eaj8 Census Counties in Colorado 2010 (64, 76)
nnpy-czzx Census County Subdivisions SF1 in Colorado 2000 (208, 73)
j

In [119]:
newdf = pd.DataFrame(
    {"s4x4":ids,
    "name":names,
    "shape":shapes,
    "data":datadict})


In [22]:
newdf.to_csv("census_summary.csv")

NameError: name 'newdf' is not defined

In [ ]:
for index,row in newdf.iterrows():
    print(row["data"].head())

In [96]:
test = pd.DataFrame(data)

In [97]:
test.to_csv("census_summary.csv")

In [29]:
pd.read_csv("https://data.colorado.gov/api/views/ph72-axst/rows.csv?accessType=DOWNLOAD")

,the_geom,geoname,geonum,pop,hispanic,white_nh,black_nh,ntvam_nh,asian_nh,hawpi_nh,other_nh,twoplus_nh,male,female,ageless5,age5_9,age10_14,age15_19,age20_24,age25_29,age30_34,age35_39,age40_44,age45_49,age50_54,age55_59,age60_64,age65_69,age70_74,age75_79,age80_84,age85pl,ageless18,age18_24,med_age,households,familyhh,nonfamhh,hhldralone,hhldr_naln,housing_un,occ_hu,vac_hu,owned,rented,pop25plus,nohsdipl,hsgrad_sc,bachl_hghr,med_hh_inc,med_fam_in,per_cap_in,med_yr_blt,med_c_rent,med_g_rent,med_hm_val,citz_birth,citz_nat,not_citz,born_in_co,brn_oth_st,ntv_b_o_us,foreign_b,pop_1p,same_house,same_cnty,same_state,diff_state,frm_abroad,wrkrs_16pl,car_all,car_alone,car_carpoo,public_trn,pt_bus,pt_other,bike,walk,tr_other,wrk_home,w_16pl_nh,t_less_10,t_10_19,t_20_29,t_30_39,t_40_59,t_60_pl,pop_3pl,enrolled,preschool,kndrgrtn,gr_1_4,gr_5_8,gr_9_12,undergrad,grad_prof,n_enrolled,blt_2010_p,b2000_2009,b1990_1999,b1980_1989,b1970_1979,b1960_1969,b1950_1959,b1940_1949,b1939_e,own_occ_hu,v_l_50k,v50k_100k,v100k_150k,v150k_200k,v200k_250k,v250k_300k,v300k_400k,v400k_500k,v500k_750k,v750k_1m,v_1m_plus,rnt_occ_hu,rntl400,r400t599,r600t799,r800t999,r1000t1249,r1250t1499,r1500t1999,r2000pl,rnocshr,ps_uni,ps_below,ps_bel150,tot_l18,puni_65pl,pov_l18,pov_65pl,avghhsize,hhi_l20k,hhi20_30,hhi30_40,hhi40_50,hhi50_60,hhi60_75,hhi75_100,hhi100_125,hhi125_150,hhi150_200,hhi200_pl,civ_ni_pop,disabled,pop16_pls,laborforce,civ_lf,emp,unemp,armedfrcs,not_lf,civ_ni_
0,"MULTIPOLYGON (((-109.060253 38.599328, -109.05...",Colorado,108,5531141,1184794,3778212,216732,30131,169556,7480,10352,133884,2780831,2750310,335199,355399,359334,354630,377987,426632,422525,390194,359083,357390,357964,363001,331165,268440,187391,121791,79953,83063,1259819,522730,36.6,2113387,1351683,761704,578532,183172,2352202,2113387,238815,1371145,742242,3748592,322718,1920982,1504892,68811,84540,36415,1983,1078,1196,313600,4988164,224034,318943,2351275,2551674,85215,542977,5467766,4464444,467576,275839,225216,34691,2822072,2374938,2125477,249461,86516,67401,19115,33829,80354,28221,218214,2603858,326205,757858,571526,453440,309029,185800,5331855,1396806,88085,70447,284619,284891,284520,306341,77903,3935049,124974,424374,392256,341166,437144,204884,179513,63720,184171,1371145,55833,49731,79507,141644,161206,160409,274043,177995,176206,53332,41239,742242,34250,33472,75720,108504,136174,117297,142127,69793,24905,5409817,590504,990886,1242689,725423,168248,54227,2.56,254215,165261,175957,168560,163943,212739,288416,212974,145127,159913,166282,5439267,575430,4409148,3012312,2978700,2839374,139326,33612,1396836,5439267


In [34]:
from itables import init_notebook_mode
init_notebook_mode(all_interactive=True)

<IPython.core.display.Javascript object>

In [23]:
df = pd.read_csv("census_summary.csv")

In [24]:
df.head(100)

,Unnamed: 0,s4x4,name,shape,data
0,0,n53p-giqe,Census Places in Colorado 2012,"(455, 157)",th...
1,1,gu6r-zvv2,Census Tracts in Colorado 2010,"(1249, 77)",th...
2,2,sn6p-34bq,Census Counties in Colorado 2016,"(64, 157)",pop ...
3,3,9cn8-g4er,Census Places in Colorado 2014,"(458, 157)",geonum ...
4,4,gafi-svtb,Census Congressional Districts in Colorado 2014,"(7, 156)",pop hispanic ...
5,5,qx2d-46a5,Census Core Based Statistical Area in Colorado...,"(14, 271)",th...
6,6,khe9-7d2c,Census Zip Codes SF1 in Colorado 2000,"(483, 70)",th...
7,7,v56b-yh3u,Census Core Based Statistical Area in Colorado...,"(14, 325)",th...
8,8,mxmm-svgz,Census County Subdivisions in Colorado 2012,"(209, 331)",th...
9,9,f3th-4ste,Census County Subdivisions SF3 in Colorado 2000,"(208, 104)",th...


In [25]:
df.shape

(71, 5)

In [38]:
ans = True
while ans :
    ans = input("enter 4x4 to remove")
    

enter 4x4 to remove 4x42
enter 4x4 to remove False
enter 4x4 to remove 


In [29]:
df.columns

RangeIndex(start=0, stop=71, step=1)

In [26]:
df.columns = ["Name","Shape","Data"]

ValueError: Length mismatch: Expected axis has 71 elements, new values have 3 elements

In [112]:
df["Data"].head()

Unnamed: 0                                                 data
n53p-giqe                                                 th...
gu6r-zvv2                                                 th...
sn6p-34bq           pop                                     ...
9cn8-g4er          geonum                                   ...
Name: Data, dtype: object

In [30]:
yes

[]

In [54]:
def getFields(w4x4=""):
    '''Reads the Inventory google sheet and getsthe fields by 4x4 dataset id and by the title'''
    bic_etl_home = os.getenv('bic_etl_home')
    print("------------")
    print("Get Fields starting",bic_etl_home)
    print("------------")

## Add the bic_etl/general/script directory to path 
    sys.path.insert(0, os.path.join(bic_etl_home, 'general', 'scripts'))

    SERVICE_ACCOUNT_FILE = os.path.join("/home","joe", 'work',"client_secret.json")
    print(SERVICE_ACCOUNT_FILE)
    scope = ['https://www.googleapis.com/auth/spreadsheets.readonly',
             "https://www.googleapis.com/auth/drive"]

    creds = ServiceAccountCredentials.from_json_keyfile_name(SERVICE_ACCOUNT_FILE,
     scope)
    client = gspread.authorize(creds)

    fields_sheet = client.open('BIC Data Inventory and Metadata').worksheet(
        'MetadataRepository')
    
    print("field sheet",fields_sheet)

    dfFields = pd.DataFrame(fields_sheet.get_all_records(head=3))
    
    fields = {}
    for index,row in dfFields.iterrows():
        s4x4 = row["Socrata Link"].strip()
        a4x4 = row["API 4x4"].strip()
        fields[s4x4] = a4x4
  
    return fields

fields = getFields()

------------
Get Fields starting /home/joe/bic_etl
------------
/home/joe/work/client_secret.json
field sheet <Worksheet 'MetadataRepository' id:323813469>


In [55]:
fields

{'rifs-n6ib': 'rifs-n6ib',
 'c8jj-hcxj': 'nigi-mnyx',
 '82s5-cpkk': '82s5-cpkk',
 'n55r-9hud': 'n55r-9hud',
 'ms6b-y4xc': 'ms6b-y4xc',
 'e6e8-qmi7': 'px7t-j3xt',
 'hxf8-ab6k': 'yt5k-hawq',
 'p8m4-v33g': 'vgwh-ygn3',
 'g53r-j5td': '6pd5-jdur',
 'jnj7-fw37': 'jnj7-fw37',
 'gyeb-jc69': '9gxp-sqz6',
 'u2t6-bfhr': 'iw7e-tqkz',
 'cjkq-q9ih': 'iaz7-gn2p',
 '4e3w-qire': 'da3r-y7xz',
 'busm-qa5b': 'vu7j-izta',
 '2cpa-vbur': 'udxx-a7kq',
 'bu8h-8sux': 'suda-myfn',
 'pt2g-89wc': '7rd3-4hgz',
 'bynd-i2hj': 'cy35-ndjp',
 'k3gg-hhc8': 'k3gg-hhc8',
 'x8tb-f3vh': 'x8tb-f3vh',
 '2yhn-3dbj': '2yhn-3dbj',
 '6kn4-89kh': '6kn4-89kh',
 '869n-zj3f': '869n-zj3f',
 'sqs8-2un5': 'sqs8-2un5',
 '3sm5-jtur': '3sm5-jtur',
 'ehk3-i5tr': 'ehk3-i5tr',
 'v9m8-x8dh': 'v9m8-x8dh',
 'j7a3-jgd3': 'j7a3-jgd3',
 'kapc-ib6e': 'ii5c-5549',
 'htyp-tqzh': 'htyp-tqzh',
 'ier5-5ms2': 'ier5-5ms2',
 'pwjb-9dd5': 'pwjb-9dd5',
 '9pwz-gi5v': 'ic4i-9zku',
 'd4s4-xqg6': '394i-tr5f',
 'msap-49q7': 'msap-49q7',
 'wz8h-dap6': 'wz8h-dap6',
 

In [50]:
SERVICE_ACCOUNT_FILE = os.path.join("/home","joe", 'work',"client_secret.json")
print(SERVICE_ACCOUNT_FILE)
scope = ['https://www.googleapis.com/auth/spreadsheets.readonly',
         "https://www.googleapis.com/auth/drive"]

creds = ServiceAccountCredentials.from_json_keyfile_name(SERVICE_ACCOUNT_FILE,
 scope)
client = gspread.authorize(creds)

fields_sheet = client.open('BIC Data Inventory and Metadata').worksheet(
    'MetadataRepository')

print("field sheet",fields_sheet)

dfFields = pd.DataFrame(fields_sheet.get_all_records(head=3))

/home/joe/work/client_secret.json
field sheet <Worksheet 'MetadataRepository' id:323813469>


In [52]:
dfFields.columns

Index(['Dataset Title', 'Short Description', 'Category', 'Keywords',
       'License Type', 'Data Provider', 'Data Provided by', 'Source Link',
       'State Steward', 'Citation', 'Agency Program Page',
       'Agency Data Series Page', 'Business Contact and Phone',
       'Technical Contact and Phone', 'Data Source', 'Unit of Analysis',
       'Granularity Coverage', 'Geographic Extent and Division',
       'Collection Mode', 'Collection Methodology',
       'Data Collection Instrument', 'Date of Initial Dataset Creation',
       'Field Names, comma delimited', 'Oldest Record in Dataset',
       'Newest Record in Dataset', 'Long Description', 'Data Dictionary',
       'Additional Metadata', 'Technical Documentation',
       'Data Quality Certification',
       'Applicable Information Quality Guideline Designation',
       'Stewardship Plan', 'Collection Method', 'Horizontal Accuracy',
       'Horizontal Coordinate System', 'Update Schedule', 'Update Method',
       'Source Update Sche

In [49]:
a[1]

['CIM Category ->',
 'General Information',
 '',
 '',
 'Licensing & Attribution',
 '',
 '',
 '',
 '',
 'Contributing Agency Information',
 '',
 '',
 '',
 '',
 '',
 'Dataset Coverage',
 '',
 '',
 'Data Description',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 'Additional Dataset Documentation',
 '',
 '',
 'Data Quality',
 '',
 '',
 'Geospatial',
 '',
 '',
 'Data Updates',
 '',
 '',
 '',
 '',
 'Semantics & RDF',
 '',
 '',
 'Internal Data',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '']

In [9]:
dfCIMMapCen[['UID', 'Name', 'Owner','Audience','Type']].to_csv("Map_Datasets_Census.csv",index=False)